In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "3"
# os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda-12.4/lib64"

In [3]:
from tqdm import tqdm

In [4]:
import gc
import os
import torch


def clear():
    """Clear GPU memory"""
    visible_devices = os.getenv("CUDA_VISIBLE_DEVICES")
    print(f"Clearing GPU memory for {visible_devices}")
    torch.cuda.empty_cache()
    gc.collect()

In [5]:
from datasets import load_dataset

ds = load_dataset("openai/gsm8k", "main", split="train[:100]").map(
    lambda x: {"prompt": x["question"], "answer": x["answer"]}
)
# ds = load_dataset("Idavidrein/gpqa", "gpqa_diamond", split="train")
# ds = ds.map(
#     lambda x: {"prompt": x["Question"], "answer": x["Correct Answer"]}, remove_columns=ds.column_names
# )
ds[0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72',
 'prompt': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'}

In [6]:
from vllm import LLM
from vllm.sampling_params import SamplingParams

# clear()

# model = 'Qwen/QwQ-32B'
# model = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
# model = "meta-llama/Llama-3.1-8B-Instruct"
# model = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
model = "bdsaglam/Llama-3.1-8B-Instruct-ragent-grpo-musique-merged"

llm = LLM(
    model=model,
    dtype="bfloat16",
    gpu_memory_utilization=0.8,
    max_model_len=8192,
    trust_remote_code=True,
)


INFO 04-20 07:44:49 __init__.py:183] Automatically detected platform cuda.
INFO 04-20 07:44:57 config.py:520] This model supports multiple tasks: {'embed', 'reward', 'classify', 'score', 'generate'}. Defaulting to 'generate'.
INFO 04-20 07:44:57 llm_engine.py:232] Initializing an LLM engine (v0.7.0) with config: model='bdsaglam/Llama-3.1-8B-Instruct-ragent-grpo-musique-merged', speculative_config=None, tokenizer='bdsaglam/Llama-3.1-8B-Instruct-ragent-grpo-musique-merged', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 04-20 07:45:04 model_runner.py:1115] Loading model weights took 14.9888 GB
INFO 04-20 07:45:06 worker.py:266] Memory profiling takes 1.00 seconds
INFO 04-20 07:45:06 worker.py:266] the current vLLM instance can use total_gpu_memory (79.26GiB) x gpu_memory_utilization (0.80) = 63.40GiB
INFO 04-20 07:45:06 worker.py:266] model weights take 14.99GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.23GiB; the rest of the memory reserved for KV Cache is 47.09GiB.
INFO 04-20 07:45:06 executor_base.py:108] # CUDA blocks: 24110, # CPU blocks: 2048
INFO 04-20 07:45:06 executor_base.py:113] Maximum concurrency for 8192 tokens per request: 47.09x
INFO 04-20 07:45:09 model_runner.py:1430] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:17<00:00,  2.04it/s]

INFO 04-20 07:45:26 model_runner.py:1558] Graph capturing finished in 17 secs, took 0.26 GiB
INFO 04-20 07:45:26 llm_engine.py:429] init engine (profile, create kv cache, warmup model) took 22.06 seconds


In [7]:
outputs = llm.chat(
    [
        {"role": "user", "content": "Hello"},
    ],
)
outputs

INFO 04-20 07:45:27 chat_utils.py:330] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  4.79it/s, est. speed input: 177.87 toks/s, output: 76.91 toks/s]


[RequestOutput(request_id=0, prompt='<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nHello<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n', prompt_token_ids=[128000, 128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 128009, 128006, 882, 128007, 271, 9906, 128009, 128006, 78191, 128007, 271], encoder_prompt=None, encoder_prompt_token_ids=None, prompt_logprobs=None, outputs=[CompletionOutput(index=0, text='Hello! Is there something I can help you with, or would you like to', token_ids=(9906, 0, 2209, 1070, 2555, 358, 649, 1520, 499, 449, 11, 477, 1053, 499, 1093, 311), cumulative_logprob=None, logprobs=None, finish_reason=length, stop_reason=None)], finished=True, metrics=RequestMetrics(arrival_time=1745124327.3097658, last_token_time=1745124327.519

In [34]:
def chat_completion(messages, max_tokens=8000, temperature=0.5, top_p=0.95, **kwargs):
    """
    Perform chat completion with the given messages and token limit.

    Args:
        messages: List of message dictionaries with role and content
        max_tokens: Maximum tokens to generate in response

    Returns:
        Generated text from the model
    """
    sampling_params = SamplingParams(
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        **kwargs,
    )
    continue_final_message = messages[-1]["role"] == "assistant"

    outputs = llm.chat(
        messages,
        sampling_params,
        continue_final_message=continue_final_message,
        add_generation_prompt=not continue_final_message,
        use_tqdm=False,
    )
    return outputs[0]

In [35]:
example = ds[10]
prompt = example["prompt"]
answer = example["answer"]
print(prompt)
print()
print("Answer: ", answer)

A deep-sea monster rises from the waters once every hundred years to feast on a ship and sate its hunger. Over three hundred years, it has consumed 847 people. Ships have been built larger over time, so each new ship has twice as many people as the last ship. How many people were on the ship the monster ate in the first hundred years?

Answer:  Let S be the number of people on the first hundred years’ ship.
The second hundred years’ ship had twice as many as the first, so it had 2S people.
The third hundred years’ ship had twice as many as the second, so it had 2 * 2S = <<2*2=4>>4S people.
All the ships had S + 2S + 4S = 7S = 847 people.
Thus, the ship that the monster ate in the first hundred years had S = 847 / 7 = <<847/7=121>>121 people on it.
#### 121


### Baseline (no control)

In [47]:
messages = [
        {"role": "user", "content": prompt},
        # {"role": "assistant", "content": "<think>\n"},
    ]

In [48]:
output = chat_completion(messages)
output

RequestOutput(request_id=38, prompt='<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nA deep-sea monster rises from the waters once every hundred years to feast on a ship and sate its hunger. Over three hundred years, it has consumed 847 people. Ships have been built larger over time, so each new ship has twice as many people as the last ship. How many people were on the ship the monster ate in the first hundred years?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n', prompt_token_ids=[128000, 128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 128009, 128006, 882, 128007, 271, 32, 5655, 7962, 64, 18118, 38268, 505, 279, 21160, 3131, 1475, 7895, 1667, 311, 53268, 389, 264, 8448, 323, 274, 349, 1202, 34906, 13, 6193, 2380, 7895, 1667, 11, 433, 706, 270

In [49]:
print("# prompt tokens: ", len(output.prompt_token_ids))
print("# completion tokens: ", len(output.outputs[0].token_ids))


# prompt tokens:  110
# completion tokens:  223


In [50]:
output.outputs[0].text

"To find the number of people on the first ship, we can use a geometric progression formula since the number of people on each ship doubles. \n\nLet's denote the number of people on the first ship as 'a'. The total number of people consumed is the sum of the geometric progression with a common ratio of 2 and 3 terms (since there are three ships).\n\nThe formula for the sum of a geometric progression is S = a * (r^n - 1) / (r - 1), where S is the sum, a is the first term, r is the common ratio, and n is the number of terms.\n\nHowever, since we are given the total number of people (847) and the common ratio (2), we can rearrange the formula to solve for 'a'.\n\n847 = a * (2^3 - 1) / (2 - 1)\n847 = a * 7\n\na = 847 / 7\na = 121\n\nSo, the number of people on the ship the monster ate in the first hundred years is 121."